# 🧠 NeuroVision 3D: 2D Brain MRI Tumor Segmentation (JPG / PNG)
### Training a 2D U-Net for Patient Image Uploads

This notebook trains a high-precision **2D U-Net** on brain MRI images (`.jpg`, `.png`, `.tif`).

- **Dataset:** Brain MRI Segmentation (`mateuszbuda/lgg-mri-segmentation`)
- **Model:** 2D U-Net with Skip Connections
- **Input:** 2D MRI Images (RGB / Grayscale JPG/PNG/TIF)
- **Output:** Exact 2D Tumor Contour Mask
- **Training Time:** ~3 to 5 minutes on a free Kaggle GPU

## Step 1: Install Dependencies & Check Environment

In [ ]:
!pip install -q "monai[all]" albumentations opencv-python matplotlib tqdm

import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

## Step 2: Discover and Match MRI Images with Tumor Masks

In [ ]:
import os
import glob
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm

# Search Kaggle input for 2D MRI datasets
all_files = glob.glob("/kaggle/input/**/*", recursive=True)
mask_files = [f for f in all_files if "_mask" in f and f.lower().endswith((".tif", ".png", ".jpg", ".jpeg"))]

pairs = []
for mask_p in mask_files:
    img_p = mask_p.replace("_mask", "")
    if os.path.exists(img_p):
        pairs.append({"image": img_p, "mask": mask_p})

print(f"Found {len(pairs)} matched 2D MRI image-mask pairs!")
if len(pairs) > 0:
    print("Example Pair:", pairs[0])
else:
    print("⚠️ Please attach the 'Brain MRI segmentation' dataset (mateuszbuda/lgg-mri-segmentation)!")

## Step 3: Dataset Class & Preprocessing Transforms

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

IMG_SIZE = 256

class BrainMRIDataset2D(Dataset):
    def __init__(self, data_pairs, is_train=True):
        self.pairs = data_pairs
        self.is_train = is_train
        
    def __len__(self):
        return len(self.pairs)
        
    def __getitem__(self, idx):
        item = self.pairs[idx]
        
        # Load image (convert to RGB)
        img = cv2.imread(item["image"], cv2.IMREAD_COLOR)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        
        # Load mask (binary 0 or 1)
        mask = cv2.imread(item["mask"], cv2.IMREAD_GRAYSCALE)
        mask = cv2.resize(mask, (IMG_SIZE, IMG_SIZE))
        mask = (mask > 127).astype(np.float32)
        
        # Simple Data Augmentation for training
        if self.is_train and np.random.rand() > 0.5:
            img = np.fliplr(img).copy()
            mask = np.fliplr(mask).copy()
            
        # Normalize to [0, 1] and convert to tensor (C, H, W)
        img_tensor = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0
        mask_tensor = torch.from_numpy(mask).unsqueeze(0).float()
        
        return img_tensor, mask_tensor

# 85% Train, 15% Validation Split
split = int(0.85 * len(pairs))
train_ds = BrainMRIDataset2D(pairs[:split], is_train=True)
val_ds = BrainMRIDataset2D(pairs[split:], is_train=False)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=2)

print(f"Training samples: {len(train_ds)} | Validation samples: {len(val_ds)}")

## Step 4: 2D U-Net Model Architecture & Loss

In [ ]:
from monai.networks.nets import UNet
from monai.losses import DiceCELoss

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# Classical 2D U-Net:
# 3 Input Channels (RGB JPG/PNG), 1 Output Channel (Tumor mask)
model_2d = UNet(
    spatial_dims=2,
    in_channels=3,
    out_channels=1,
    channels=(16, 32, 64, 128, 256),
    strides=(2, 2, 2, 2),
    num_res_units=2,
).to(device)

# Compound Loss: Dice Loss + Binary Cross Entropy (BCE)
loss_func = DiceCELoss(sigmoid=True)
optimizer = torch.optim.AdamW(model_2d.parameters(), lr=1e-3, weight_decay=1e-5)

print(f"2D U-Net instantiated on {device}.")

## Step 5: Fast Training Loop (10 Epochs ~ 3 Minutes)

In [ ]:
import time
EPOCHS = 10
best_val_loss = float("inf")
os.makedirs("/kaggle/working/checkpoints_2d", exist_ok=True)
best_model_path = "/kaggle/working/checkpoints_2d/neurovision_2d_unet.pth"

start_time = time.time()
print(f"Training 2D U-Net for {EPOCHS} epochs...")

for epoch in range(EPOCHS):
    model_2d.train()
    train_loss = 0.0
    
    for imgs, masks in tqdm(train_loader, desc=f"Epoch {epoch + 1}/{EPOCHS}"):
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        
        outputs = model_2d(imgs)
        loss = loss_func(outputs, masks)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        
    train_loss /= len(train_loader)
    
    # Validation
    model_2d.eval()
    val_loss = 0.0
    with torch.no_grad():
        for v_imgs, v_masks in val_loader:
            v_imgs, v_masks = v_imgs.to(device), v_masks.to(device)
            v_outs = model_2d(v_imgs)
            val_loss += loss_func(v_outs, v_masks).item()
            
    val_loss /= len(val_loader)
    print(f"Epoch {epoch + 1}: Train Loss = {train_loss:.4f} | Val Loss = {val_loss:.4f}")
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model_2d.state_dict(), best_model_path)
        print(f"🔥 Saved new best 2D model (Val Loss: {val_loss:.4f})")

print(f"Training complete in {(time.time() - start_time) / 60:.2f} minutes!")

## Step 6: Test Inference on Sample Patient JPG

In [ ]:
# Load best weights
model_2d.load_state_dict(torch.load(best_model_path))
model_2d.eval()

# Pick a validation image that has a tumor
for test_img, test_mask in val_loader:
    if test_mask.sum() > 500:
        break

with torch.no_grad():
    pred_raw = model_2d(test_img.to(device))
    pred_binary = (torch.sigmoid(pred_raw) > 0.5).float().cpu().numpy()[0, 0]

orig = test_img[0].permute(1, 2, 0).numpy()
gt = test_mask[0, 0].numpy()

# Plot Original JPG, Ground Truth, and AI Prediction
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(orig)
axes[0].set_title("Uploaded Patient MRI (JPG/PNG)")
axes[0].axis("off")

axes[1].imshow(orig)
axes[1].imshow(gt, cmap="autumn", alpha=0.5)
axes[1].set_title("Ground Truth Tumor Area")
axes[1].axis("off")

axes[2].imshow(orig)
axes[2].imshow(pred_binary, cmap="cool", alpha=0.6)
axes[2].set_title("AI 2D U-Net Prediction")
axes[2].axis("off")

plt.tight_layout()
plt.savefig("/kaggle/working/2d_prediction_sample.png", dpi=300)
plt.show()

print("✅ 2D Model weights ready: /kaggle/working/checkpoints_2d/neurovision_2d_unet.pth")